PROJEKT CZ. 2: UCZENIE MASZYNOWE

In [33]:
!pip install pandas numpy scikit-learn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.svm import SVC, SVR
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
import warnings

warnings.filterwarnings('ignore')


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\filip\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


WCZYTANIE I PRZYGOTOWANIE DANYCH

In [34]:


df = pd.read_csv('../../data/credit_risk_dataset.csv')

# Usuwanie anomalii
df = df[df['person_age'] < 100]
df = df[df['person_emp_length'] < 60]

# Uzupełnienie braków i kodowanie zmiennych kategorycznych
df = df.fillna(df.mean(numeric_only=True))
df = pd.get_dummies(df, drop_first=True)
df = df.astype(float)

# Zmienne docelowe:
#   - klasyfikacja: loan_status (0/1 - czy pożyczka spłacona)
#   - regresja:     loan_int_rate (wysokość oprocentowania)
TARGET_CLF = 'loan_status'
TARGET_REG = 'loan_int_rate'

y_clf = df[TARGET_CLF].values.ravel()
y_reg = df[TARGET_REG].values.ravel()

# Usuwamy OBIE kolumny docelowe z X (uniknięcie target leakage)
X_clf = df.drop([TARGET_CLF, TARGET_REG], axis=1).values
X_reg = df.drop([TARGET_REG, TARGET_CLF], axis=1).values

# Podział 70% train / 30% test
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.3, stratify=y_clf, random_state=42)
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.3, random_state=42)

# Skalowanie (tylko kolumny numeryczne – bez 0/1)
binary_cols = [i for i in range(X_train_clf.shape[1])
               if np.isin(X_train_clf[:, i], [0, 1]).all()]
numeric_cols = [i for i in range(X_train_clf.shape[1])
                if i not in binary_cols]

DANE

In [35]:
df.head(5)

,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_cred_hist_length,person_home_ownership_OTHER,person_home_ownership_OWN,...,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE,loan_grade_B,loan_grade_C,loan_grade_D,loan_grade_E,loan_grade_F,loan_grade_G,cb_person_default_on_file_Y
1,21.0,9600.0,5.0,1000.0,11.14,0.0,0.10,2.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,25.0,9600.0,1.0,5500.0,12.87,1.0,0.57,3.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,23.0,65500.0,4.0,35000.0,15.23,1.0,0.53,2.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,24.0,54400.0,8.0,35000.0,14.27,1.0,0.55,4.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
5,21.0,9900.0,2.0,2500.0,7.14,1.0,0.25,2.0,0.0,1.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


KLASYFIKACJA

In [36]:
scaler_clf = StandardScaler()
X_train_clf_s = X_train_clf.copy()
X_test_clf_s = X_test_clf.copy()
X_train_clf_s[:, numeric_cols] = scaler_clf.fit_transform(X_train_clf[:, numeric_cols])
X_test_clf_s[:, numeric_cols] = scaler_clf.transform(X_test_clf[:, numeric_cols])

# ===== REGRESJA =====
binary_cols_reg = [i for i in range(X_train_reg.shape[1])
                   if np.isin(X_train_reg[:, i], [0, 1]).all()]
numeric_cols_reg = [i for i in range(X_train_reg.shape[1])
                    if i not in binary_cols_reg]

scaler_reg = StandardScaler()
X_train_reg_s = X_train_reg.copy()
X_test_reg_s = X_test_reg.copy()
X_train_reg_s[:, numeric_cols_reg] = scaler_reg.fit_transform(X_train_reg[:, numeric_cols_reg])
X_test_reg_s[:, numeric_cols_reg] = scaler_reg.transform(X_test_reg[:, numeric_cols_reg])

print("Dane przygotowane.")
print(f"  Rozmiar zbioru uczącego  (clf): {X_train_clf_s.shape}")
print(f"  Rozmiar zbioru testowego (clf): {X_test_clf_s.shape}")
print(f"  Rozmiar zbioru uczącego  (reg): {X_train_reg_s.shape}")
print(f"  Rozmiar zbioru testowego (reg): {X_test_reg_s.shape}")

Dane przygotowane.
  Rozmiar zbioru uczącego  (clf): (22175, 21)
  Rozmiar zbioru testowego (clf): (9504, 21)
  Rozmiar zbioru uczącego  (reg): (22175, 21)
  Rozmiar zbioru testowego (reg): (9504, 21)


POMOCNICZE FUNKCJE

In [37]:
def print_header(title):
    print(f"\n{'=' * 60}")
    print(f"  {title}")
    print(f"{'=' * 60}")

def print_param_header(desc):
    print(f"\n--- {desc} ---")

PROBLEM KLASYFIKACYJNY

In [38]:
print_header("PROBLEM KLASYFIKACYJNY  |  Metryka: Accuracy")


  PROBLEM KLASYFIKACYJNY  |  Metryka: Accuracy


KNN


KNN – PARAMETR 1: liczba sąsiadów (n_neighbors)

In [ ]:
print_param_header("KNN | PARAMETR 2: wagi sąsiadów (weights)")
print(f"  {'weights':<15} {'Train Acc':>12} {'Test Acc':>12}")
print(f"  {'-'*15} {'-'*12} {'-'*12}")
for w in ['uniform', 'distance']:
    m = KNeighborsClassifier(n_neighbors=5, weights=w)
    m.fit(X_train_clf_s, y_train_clf)
    tr = accuracy_score(y_train_clf, m.predict(X_train_clf_s))
    te = accuracy_score(y_test_clf, m.predict(X_test_clf_s))
    print(f"  {w:<15} {tr:>12.4f} {te:>12.4f}")

KNN – PARAMETR 2: wagi sąsiadów (weights)

In [39]:
print_param_header("KNN | PARAMETR 2: wagi sąsiadów (weights)")
print(f"  {'weights':<15} {'Train Acc':>12} {'Test Acc':>12}")
print(f"  {'-'*15} {'-'*12} {'-'*12}")
for w in ['uniform', 'distance']:
    m = KNeighborsClassifier(n_neighbors=5, weights=w)
    m.fit(X_train_clf_s, y_train_clf)
    tr = accuracy_score(y_train_clf, m.predict(X_train_clf_s))
    te = accuracy_score(y_test_clf, m.predict(X_test_clf_s))
    print(f"  {w:<15} {tr:>12.4f} {te:>12.4f}")


--- KNN | PARAMETR 2: wagi sąsiadów (weights) ---
  weights            Train Acc     Test Acc
  --------------- ------------ ------------
  uniform               0.9130       0.8847
  distance              1.0000       0.8843


SVM


SVM – PARAMETR 1: parametr regularyzacji C

In [40]:
print_param_header("SVR | PARAMETR 1: parametr regularyzacji (C)")
print(f"  {'C':<15} {'Train MSE':>12} {'Test MSE':>12}")
print(f"  {'-'*15} {'-'*12} {'-'*12}")
for c in [0.1, 1.0, 10.0, 100.0]:
    m = SVR(C=c, kernel='rbf')
    m.fit(X_train_reg_s, y_train_reg)
    tr = mean_squared_error(y_train_reg, m.predict(X_train_reg_s))
    te = mean_squared_error(y_test_reg, m.predict(X_test_reg_s))
    print(f"  C = {c:<11} {tr:>12.4f} {te:>12.4f}")


--- SVR | PARAMETR 1: parametr regularyzacji (C) ---
  C                  Train MSE     Test MSE
  --------------- ------------ ------------
  C = 0.1               3.0440       2.9741
  C = 1.0               1.9186       2.0431
  C = 10.0              1.4842       1.8785
  C = 100.0             1.2469       2.0985


SVR – PARAMETR 2: rodzaj jądra (kernel)

In [ ]:
print_param_header("SVR | PARAMETR 2: rodzaj jądra (kernel)")
print(f"  {'kernel':<15} {'Train MSE':>12} {'Test MSE':>12}")
print(f"  {'-'*15} {'-'*12} {'-'*12}")
for kern in ['linear', 'poly', 'rbf', 'sigmoid']:
    m = SVR(C=1.0, kernel=kern)
    m.fit(X_train_reg_s, y_train_reg)
    tr = mean_squared_error(y_train_reg, m.predict(X_train_reg_s))
    te = mean_squared_error(y_test_reg, m.predict(X_test_reg_s))
    print(f"  {kern:<15} {tr:>12.4f} {te:>12.4f}")


--- SVR | PARAMETR 2: rodzaj jądra (kernel) ---
  kernel             Train MSE     Test MSE
  --------------- ------------ ------------
  linear                1.7065       1.7378
  poly                  2.5846       2.8767
  rbf                   1.9186       2.0431


DRZZEWO DECYZYJNE

DRZEWO – PARAMETR 1: maksymalna głębokość (max_depth)

In [ ]:
print_param_header("Drzewo Decyzyjne | PARAMETR 1: maksymalna głębokość (max_depth)")
print(f"  {'max_depth':<15} {'Train MSE':>12} {'Test MSE':>12}")
print(f"  {'-'*15} {'-'*12} {'-'*12}")
for depth in [3, 5, 10, None]:
    m = DecisionTreeRegressor(max_depth=depth, random_state=42)
    m.fit(X_train_reg_s, y_train_reg)
    tr = mean_squared_error(y_train_reg, m.predict(X_train_reg_s))
    te = mean_squared_error(y_test_reg, m.predict(X_test_reg_s))
    label = str(depth) if depth is not None else 'None (brak)'
    print(f"  {label:<15} {tr:>12.4f} {te:>12.4f}")

DRZEWO – PARAMETR 2: min. liczba obserwacji w liściu (min_samples_leaf)

In [ ]:
print_param_header("Drzewo Decyzyjne | PARAMETR 2: min. liczba obserwacji w liściu (min_samples_leaf)")
print(f"  {'min_samples_leaf':<20} {'Train Acc':>12} {'Test Acc':>12}")
print(f"  {'-'*20} {'-'*12} {'-'*12}")
for msl in [1, 5, 20, 50]:
    m = DecisionTreeClassifier(min_samples_leaf=msl, random_state=42)
    m.fit(X_train_clf_s, y_train_clf)
    tr = accuracy_score(y_train_clf, m.predict(X_train_clf_s))
    te = accuracy_score(y_test_clf, m.predict(X_test_clf_s))
    print(f"  {msl:<20} {tr:>12.4f} {te:>12.4f}")

LAS LOSOWY

LAS LOSOWY – PARAMETR 1: liczba drzew (n_estimators)

In [ ]:
print_param_header("Las Losowy | PARAMETR 1: liczba drzew (n_estimators)")
print(f"  {'n_estimators':<15} {'Train Acc':>12} {'Test Acc':>12}")
print(f"  {'-'*15} {'-'*12} {'-'*12}")
for n_est in [10, 50, 100, 200]:
    m = RandomForestClassifier(n_estimators=n_est, random_state=42, n_jobs=-1)
    m.fit(X_train_clf_s, y_train_clf)
    tr = accuracy_score(y_train_clf, m.predict(X_train_clf_s))
    te = accuracy_score(y_test_clf, m.predict(X_test_clf_s))
    print(f"  {n_est:<15} {tr:>12.4f} {te:>12.4f}")

LAS LOSOWY – PARAMETR 2: maksymalna głębokość (max_depth)

In [ ]:
print_param_header("Las Losowy | PARAMETR 2: maksymalna głębokość drzew (max_depth)")
print(f"  {'max_depth':<15} {'Train Acc':>12} {'Test Acc':>12}")
print(f"  {'-'*15} {'-'*12} {'-'*12}")
for depth in [3, 5, 10, None]:
    m = RandomForestClassifier(n_estimators=100, max_depth=depth,
                               random_state=42, n_jobs=-1)
    m.fit(X_train_clf_s, y_train_clf)
    tr = accuracy_score(y_train_clf, m.predict(X_train_clf_s))
    te = accuracy_score(y_test_clf, m.predict(X_test_clf_s))
    label = str(depth) if depth is not None else 'None (brak)'
    print(f"  {label:<15} {tr:>12.4f} {te:>12.4f}")
